In [1]:
import numpy as np
import copy

import kuhn_poker

## Kuhn Poker

In [2]:
table = kuhn_poker.KuhnPoker()

table.play("b")
table.play("c")

(True, 'JKbc', array([ 1, -1]))

In [15]:
table.start()
table.play("c")
table.play("c")

(True, 'JKcc', array([-1,  1]))

In [16]:
table.start()
table.play("c")
table.play("b")
table.play("c")

(True, 'JKcbc', array([-1,  1]))

In [18]:
table.start()
table.play("b")
table.play("b")

(True, 'JKbb', array([-2,  2]))

## Vanilla CFR

In [23]:
def vanilla_cfr(steps):
    hands = ["JQ", "JK", "QJ", "QK", "KJ", "KQ"]
    sigma = np.zeros((2, 12, 2))

    for step in steps:
        u = np.array((2, 12, 2))
        R = np.array((2, 12, 2))

        for hand in hands:
            table = kuhn_poker.KuhnPoker(hand)
            full_tree_recursion(u, R, sigma, infoset, table)

        R /= steps       

def full_tree_recursion(u, R, sigma, infoset, reach_proba, table):
    player = table.plr_idx
    avg_utility = np.zeros(2)
    
    for action in table.actions:
        a = 0 if action == "b" else 1
        action_prob = sigma[player, infoset, a]

        next_table = copy.deepcopy(table)
        terminate, history, utility = next_table.play(action)
        
        if terminate:
            action_utility = utility
        else:
            next_infoset = next_table.get_infoset(next_table.plr_idx)[0]
            action_utility = full_tree_recursion(u, R, sigma, next_infoset, reach_proba, next_table)

        u[:, infoset, a] += action_utility
        avg_utility += action_prob * action_utility

    R[player, infoset, :] = action_prob * (u[player, infoset, :] - avg_utility[player])

    return avg_utility

In [32]:
u = np.array((2, 12, 2))
R = np.array((2, 12, 2))
sigma = 0.5 * np.ones((2, 12, 2))

table = kuhn_poker.KuhnPoker("QJ")
full_tree_recursion(u, R, sigma, table.get_infoset(0), 1/6, table)

array([[[0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5]],

       [[0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5],
        [0.5, 0.5]]])